# Explainable classification of MTSK research — final model and SHAP analysis

**Authors:** Carolina Rojas Celis · Luz Maricel Elorreaga Rodríguez · Héctor Javier Hortúa
**Institution:** Universidad El Bosque

This notebook trains the final `intfloat/multilingual-e5-large` classifier on the **first fold of the stratified 5-fold cross-validation** (see `MTSK_validacion_cruzada.ipynb`), evaluates it on that fold's held-out test documents, and runs the SHAP interpretability analysis on the same test documents.

**Protocol**
- Training, internal validation (early stopping / epoch selection) and test documents are taken from `folds.json`, produced by the cross-validation notebook. Related versions of the same work (column `Grupo`) never fall on different sides of the split.
- The test documents are used **only** for final evaluation and SHAP; never for training or epoch selection.
- Same configuration as in cross-validation: prefix `query:`, title + abstract + keywords, 128 tokens, first-token ([CLS]) pooling, Dropout(0.3) + Linear, class-weighted cross-entropy, AdamW (lr = 5e-5), 10 % linear warm-up, batch 16, max. 20 epochs, patience 3. **All backbone weights are updated during fine-tuning.**
- Seed 7 (the seed used for fold 0 in cross-validation), so this run should approximately reproduce fold 0 of the cross-validation (small differences can arise from GPU non-determinism).

**Data availability.** The corpus is not included in this repository because of copyright restrictions on abstracts. Persistent identifiers and expert labels are deposited on Zenodo.

## 1 · Installation and imports

In [ ]:
!pip install -q transformers scikit-learn openpyxl matplotlib seaborn shap huggingface_hub

In [ ]:
import os, gc, json, random, pickle
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import shap
import transformers, sklearn
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| sklearn", sklearn.__version__, "| shap", shap.__version__)

## 2 · Configuration

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

BASE_DIR  = "/content/drive/MyDrive/TG Maestria"
DATA_PATH = f"{BASE_DIR}/corpus_MTSK_depurado.xlsx"   # SAME file used in cross-validation
CV_DIR    = f"{BASE_DIR}/outputs_cv"                  # folder with folds.json from cross-validation
OUT_DIR   = f"{BASE_DIR}/outputs_modelo_final"
os.makedirs(OUT_DIR, exist_ok=True)

TEXT_COLS = ("Título", "Resumen (texto literal)", "Palabras clave (texto literal)")
LABEL_COL = "Clasificacion tematica"
LANG_COL  = "Idioma"

FOLD       = 0
SEED       = 7          # = SEED + FOLD in the cross-validation notebook
MODEL_NAME = "intfloat/multilingual-e5-large"
PREFIX     = "query: "
MAX_LEN    = 128
BATCH_SIZE = 16
LR         = 5e-5
MAX_EPOCHS = 20
PATIENCE   = 3
DROPOUT    = 0.3
WARMUP     = 0.10

N_SHAP        = None    # None = explain ALL test documents; set e.g. 20 if it takes too long
UPLOAD_TO_HUB = False   # set True only when you want to replace the model on Hugging Face
HF_REPO       = "crojasc2/mtsk-classifier"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 3 · Data and split (fold 0 of cross-validation)

In [ ]:
df = pd.read_excel(DATA_PATH).reset_index(drop=True)

comments = (PREFIX +
            df[TEXT_COLS[0]].fillna("").astype(str) + " " +
            df[TEXT_COLS[1]].fillna("").astype(str) + " " +
            df[TEXT_COLS[2]].fillna("").astype(str)).str.strip().tolist()

nam_labels = sorted(df[LABEL_COL].unique().tolist())
num_labels = len(nam_labels)
label2idx  = {l: i for i, l in enumerate(nam_labels)}
idx2label  = {i: l for i, l in enumerate(nam_labels)}
labels     = [label2idx[l] for l in df[LABEL_COL]]

folds = json.load(open(f"{CV_DIR}/folds.json"))
F = folds[FOLD]
all_idx = F["train"] + F["val"] + F["test"]
assert len(all_idx) == len(df) and len(set(all_idx)) == len(df), \
    "folds.json does not match this corpus: use exactly the same file as in cross-validation"

train_idx, val_idx, test_idx = F["train"], F["val"], F["test"]
take = lambda seq, idx: [seq[i] for i in idx]
train_comments, train_labels = take(comments, train_idx), take(labels, train_idx)
val_comments,   val_labels   = take(comments, val_idx),   take(labels, val_idx)
test_comments,  test_labels  = take(comments, test_idx),  take(labels, test_idx)

print(f"Documents: {len(df)} | classes: {nam_labels}")
print(f"Train: {len(train_idx)} | Internal validation (early stopping): {len(val_idx)} | Test: {len(test_idx)}")
print("Test per class:", pd.Series(take(df[LABEL_COL].tolist(), test_idx)).value_counts().sort_index().to_dict())

## 4 · Model

In [ ]:
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class TextDS(Dataset):
    def __init__(self, enc, labels):
        self.enc, self.labels = enc, labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(int(self.labels[i]))
        return item

class MTSKClassifier(torch.nn.Module):
    # [CLS] (first token) pooling → Dropout → Linear. Backbone weights are NOT frozen.
    def __init__(self, backbone, num_labels, p=DROPOUT):
        super().__init__()
        self.backbone   = backbone
        self.dropout    = torch.nn.Dropout(p)
        self.classifier = torch.nn.Linear(backbone.config.hidden_size, num_labels)
    def forward(self, input_ids, attention_mask):
        h = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(h))

@torch.no_grad()
def predict_proba(model, loader):
    model.eval(); out = []
    for b in loader:
        logits = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
        out.append(torch.softmax(logits, dim=-1).cpu().numpy())
    return np.concatenate(out)

# Same order of operations as run_fold() in the cross-validation notebook
set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
enc = lambda texts: tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN)
g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(TextDS(enc(train_comments), train_labels), batch_size=BATCH_SIZE, shuffle=True, generator=g)
val_loader   = DataLoader(TextDS(enc(val_comments),   val_labels),   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(TextDS(enc(test_comments),  test_labels),  batch_size=BATCH_SIZE, shuffle=False)

class_weights = compute_class_weight("balanced", classes=np.arange(num_labels), y=np.array(train_labels))
weight_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
print("Class weights:", dict(zip(nam_labels, class_weights.round(3))))

model = MTSKClassifier(AutoModel.from_pretrained(MODEL_NAME), num_labels).to(DEVICE)
print("Trainable parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 5 · Training with early stopping on the internal validation set

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR)
total     = len(train_loader) * MAX_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(total * WARMUP), total)
CKPT = "best_model.pt"

best_f1, best_epoch, bad = -1.0, 0, 0
history = {"epoch": [], "train_loss": [], "val_macro_f1": [], "val_accuracy": []}
for ep in range(1, MAX_EPOCHS + 1):
    model.train(); tot = 0.0
    for b in train_loader:
        optimizer.zero_grad()
        loss = torch.nn.functional.cross_entropy(
            model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE)),
            b["labels"].to(DEVICE), weight=weight_tensor)
        loss.backward(); optimizer.step(); scheduler.step(); tot += loss.item()
    pv  = predict_proba(model, val_loader).argmax(1)
    vf1 = f1_score(val_labels, pv, average="macro", zero_division=0)
    vac = accuracy_score(val_labels, pv)
    for k, v in zip(history, [ep, tot / len(train_loader), vf1, vac]):
        history[k].append(v)
    print(f"Epoch {ep:02d} | loss {tot/len(train_loader):.4f} | val macro-F1 {vf1:.4f} | val acc {vac:.4f}")
    if vf1 > best_f1:
        best_f1, best_epoch, bad = vf1, ep, 0
        torch.save(model.state_dict(), CKPT)
    else:
        bad += 1
        if bad >= PATIENCE:
            print(f"Early stopping at epoch {ep}")
            break

model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()
print(f"Selected epoch: {best_epoch} (internal validation macro-F1 = {best_f1:.4f})")
pd.DataFrame(history).to_csv(f"{OUT_DIR}/training_history.csv", index=False)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(history["epoch"], history["train_loss"], marker="o", color="#C0392B")
ax1.set_title("Training loss (class-weighted cross-entropy)"); ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax2.plot(history["epoch"], history["val_macro_f1"], marker="o", label="Macro-F1", color="#2E86C1")
ax2.plot(history["epoch"], history["val_accuracy"], marker="s", label="Accuracy", color="#E67E22")
ax2.set_title("Internal validation metrics"); ax2.set_xlabel("Epoch"); ax2.set_ylim(0, 1); ax2.legend()
for ax in (ax1, ax2):
    ax.axvline(best_epoch, ls="--", color="#27AE60", lw=1.2)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True)); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/training_curves.png", dpi=300, bbox_inches="tight"); plt.show()

## 6 · Evaluation on the held-out test documents

In [ ]:
test_probs = predict_proba(model, test_loader)
test_pred  = test_probs.argmax(1)
y_test     = np.array(test_labels)

print(classification_report(y_test, test_pred, target_names=nam_labels, digits=4, zero_division=0))
summary = {"fold": FOLD, "seed": SEED, "n_test": int(len(y_test)), "best_epoch": best_epoch,
           "val_macro_f1": best_f1,
           "accuracy":  accuracy_score(y_test, test_pred),
           "macro_f1":  f1_score(y_test, test_pred, average="macro", zero_division=0),
           "precision": precision_score(y_test, test_pred, average="macro", zero_division=0),
           "recall":    recall_score(y_test, test_pred, average="macro", zero_division=0)}
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in summary.items()})
json.dump(summary, open(f"{OUT_DIR}/test_summary.json", "w"), indent=2)
pd.DataFrame(classification_report(y_test, test_pred, target_names=nam_labels, digits=4,
             output_dict=True, zero_division=0)).T.to_csv(f"{OUT_DIR}/test_per_class.csv")

# Consistency check against fold 0 of the cross-validation (if available)
oof_path = f"{CV_DIR}/e5-large_predicciones_oof.csv"
if os.path.exists(oof_path):
    oof = pd.read_csv(oof_path)
    o = oof[oof["pliegue"] == FOLD]
    cv_f1 = f1_score(o["etiqueta"], o["prediccion"], average="macro", zero_division=0)
    print(f"Cross-validation fold {FOLD} macro-F1 = {cv_f1:.4f} | this run = {summary['macro_f1']:.4f}")

In [ ]:
# Prediction table with ALL class probabilities (source for every number cited in the qualitative analysis)
pred_table = pd.DataFrame(test_probs, columns=[f"P_{l}" for l in nam_labels])
pred_table.insert(0, "doc", range(len(test_idx)))            # position within the test set (= index in SHAP)
pred_table.insert(1, "row_in_corpus", test_idx)              # row in corpus_MTSK_depurado.xlsx (0-based)
if "ID original" in df.columns:
    pred_table.insert(2, "ID original", df.loc[test_idx, "ID original"].values)
pred_table["title"]      = df.loc[test_idx, TEXT_COLS[0]].astype(str).str.replace("\n", " ").values
pred_table["language"]   = df.loc[test_idx, LANG_COL].values if LANG_COL in df.columns else ""
pred_table["true"]       = [idx2label[i] for i in y_test]
pred_table["predicted"]  = [idx2label[i] for i in test_pred]
pred_table["confidence"] = test_probs.max(1)
pred_table["correct"]    = pred_table["true"] == pred_table["predicted"]
pred_table.to_csv(f"{OUT_DIR}/test_predictions.csv", index=False)
pred_table[["doc", "true", "predicted", "confidence", "correct"]]

In [ ]:
cm  = confusion_matrix(y_test, test_pred, labels=range(num_labels))
cmn = cm / cm.sum(axis=1, keepdims=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.heatmap(cm,  annot=True, fmt="d",   cmap="Blues", xticklabels=nam_labels, yticklabels=nam_labels, ax=axes[0])
sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
            xticklabels=nam_labels, yticklabels=nam_labels, ax=axes[1])
axes[0].set_title("Confusion matrix (counts)"); axes[1].set_title("Confusion matrix (row-normalised)")
for ax in axes:
    ax.set_xlabel("Predicted label"); ax.set_ylabel("True label")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/test_confusion_matrix.png", dpi=300, bbox_inches="tight"); plt.show()

## 7 · SHAP interpretability

`shap.Explainer` with the tokenizer-based `Text` masker (Partition explainer). No background dataset is used: masked tokens are replaced by the tokenizer's mask token. Attributions are computed on the class **probabilities** (softmax outputs) for the held-out test documents.

In [ ]:
class SHAPWrapper:
    def __init__(self, model, tokenizer):
        self.model, self.tokenizer = model, tokenizer
    def __call__(self, texts):
        texts = [texts] if isinstance(texts, str) else list(texts)
        e = self.tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        self.model.eval()
        with torch.no_grad():
            logits = self.model(input_ids=e["input_ids"], attention_mask=e["attention_mask"])
        return torch.softmax(logits, dim=-1).cpu().numpy()

wrapper   = SHAPWrapper(model, tokenizer)
explainer = shap.Explainer(wrapper, masker=shap.maskers.Text(tokenizer), output_names=nam_labels)

n_explain   = len(test_comments) if N_SHAP is None else min(N_SHAP, len(test_comments))
shap_values = explainer(test_comments[:n_explain])
print("SHAP values:", shap_values.shape, "| documents explained:", n_explain)
with open(f"{OUT_DIR}/shap_values.pkl", "wb") as f:
    pickle.dump(shap_values, f)

In [ ]:
# Global importance per output class: mean |SHAP| of each subtoken across the explained documents
# (documents in which the subtoken does not appear count as 0). Saved as CSV + figure.
rows = []
for c, lab in enumerate(nam_labels):
    acc = {}
    for d in range(n_explain):
        toks = shap_values[d].data
        vals = shap_values[d].values[:, c]
        for t, v in zip(toks, vals):
            t = str(t).strip()
            if t:
                acc[t] = acc.get(t, 0.0) + abs(float(v))
    total = sum(acc.values())
    for t, s in acc.items():
        rows.append({"class": lab, "subtoken": t, "mean_abs_shap": s / n_explain, "share_of_total": s / total})
glob = pd.DataFrame(rows).sort_values(["class", "mean_abs_shap"], ascending=[True, False])
glob.to_csv(f"{OUT_DIR}/shap_global_importance.csv", index=False)

TOP = 10
fig, axes = plt.subplots(1, num_labels, figsize=(4.2 * num_labels, 4.5))
for ax, lab in zip(axes, nam_labels):
    sub = glob[glob["class"] == lab]
    top = sub.head(TOP)
    rest = sub["share_of_total"].iloc[TOP:].sum()
    ax.barh(top["subtoken"][::-1], top["mean_abs_shap"][::-1], color="#2E86C1")
    ax.set_title(f"{lab}  (remaining subtokens: {rest:.2f} of total)", fontsize=10)
    ax.set_xlabel("Mean |SHAP value|")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/shap_global_importance.png", dpi=300, bbox_inches="tight"); plt.show()
glob.groupby("class").head(TOP)

In [ ]:
# Document-level cases. By default: every misclassified test document + the most confident correct one per class.
# To choose other documents, set CASES = [list of 'doc' values from test_predictions.csv].
CASES = None
explained = pred_table.iloc[:n_explain]
if CASES is None:
    errors  = explained[~explained["correct"]]["doc"].tolist()
    correct = (explained[explained["correct"]].sort_values("confidence", ascending=False)
               .groupby("true").head(1)["doc"].tolist())
    CASES = errors + correct

os.makedirs(f"{OUT_DIR}/cases", exist_ok=True)
for d in CASES:
    r = pred_table.iloc[d]
    tag = "correct" if r["correct"] else "error"
    print(f"\n── Doc {d} [{tag}] true={r['true']} predicted={r['predicted']} confidence={r['confidence']:.3f}")
    print("   " + " | ".join(f"{l}={r['P_'+l]:.3f}" for l in nam_labels))
    classes = [r["predicted"]] if r["correct"] else [r["predicted"], r["true"]]
    for cls in classes:
        shap.plots.bar(shap_values[d, :, cls], max_display=15, show=False)
        plt.title(f"Doc {d} — contributions to {cls} (true {r['true']}, predicted {r['predicted']})", fontsize=10)
        plt.tight_layout()
        plt.savefig(f"{OUT_DIR}/cases/doc{d}_{tag}_toward_{cls}.png", dpi=300, bbox_inches="tight")
        plt.show()
    html = shap.plots.text(shap_values[d], display=False)
    open(f"{OUT_DIR}/cases/doc{d}_{tag}_text.html", "w", encoding="utf-8").write(html)
print("\nCases saved in", f"{OUT_DIR}/cases")

## 8 · Export to Hugging Face Hub (optional)

In [ ]:
HF_DIR = "hf_model"
os.makedirs(HF_DIR, exist_ok=True)
model.backbone.save_pretrained(HF_DIR)
tokenizer.save_pretrained(HF_DIR)
head = model.classifier.state_dict()
torch.save({
    "classification_head": head,        # key used by earlier versions of the Space
    "classifier_state_dict": head,      # same weights, alternative key
    "label2idx": label2idx, "idx2label": idx2label, "nam_labels": nam_labels,
    "num_labels": num_labels, "model_name": MODEL_NAME, "prefix": PREFIX,
    "max_length": MAX_LEN, "dropout": DROPOUT, "seed": SEED, "fold": FOLD,
}, os.path.join(HF_DIR, "classifier_head.pt"))

langs = sorted(df[LANG_COL].dropna().unique().tolist()) if LANG_COL in df.columns else ["es", "en", "pt"]
readme = f'''---
language: {json.dumps(langs)}
tags: [text-classification, mtsk, mathematics-education, multilingual]
license: mit
metrics: [f1, accuracy]
---

# MTSK thematic classifier

Classifies research on Mathematics Teacher's Specialised Knowledge (MTSK) into the five thematic areas of the RedMTSK (T1–T5), from title, abstract and keywords.

- Base model: `{MODEL_NAME}`, fully fine-tuned, [CLS] pooling + Dropout({DROPOUT}) + Linear.
- Input: `"{PREFIX}" + title + " " + abstract + " " + keywords`, truncated to {MAX_LEN} tokens.
- Training data: {len(df)} expert-labelled documents (languages: {", ".join(langs)}); this checkpoint was trained on fold {FOLD} of a stratified 5-fold cross-validation.
- Held-out test (fold {FOLD}, n = {summary["n_test"]}): macro-F1 = {summary["macro_f1"]:.4f}, accuracy = {summary["accuracy"]:.4f}. See the article for cross-validated results.

The classification head is stored in `classifier_head.pt`.
'''
open(os.path.join(HF_DIR, "README.md"), "w", encoding="utf-8").write(readme)
print(readme)

if UPLOAD_TO_HUB:
    from google.colab import userdata
    from huggingface_hub import login, HfApi, list_repo_files
    login(userdata.get("huggingface"))
    api = HfApi()
    api.create_repo(HF_REPO, exist_ok=True, repo_type="model")
    api.upload_folder(folder_path=HF_DIR, repo_id=HF_REPO, repo_type="model")
    print("Uploaded:", list_repo_files(HF_REPO))
else:
    print("UPLOAD_TO_HUB = False → model not uploaded")

In [ ]:
json.dump({"n_docs": len(df), "fold": FOLD, "seed": SEED, "model": MODEL_NAME, "prefix": PREFIX,
           "max_len": MAX_LEN, "batch_size": BATCH_SIZE, "lr": LR, "max_epochs": MAX_EPOCHS,
           "patience": PATIENCE, "dropout": DROPOUT, "warmup": WARMUP, "n_shap": n_explain,
           "torch": torch.__version__, "transformers": transformers.__version__,
           "sklearn": sklearn.__version__, "shap": shap.__version__},
          open(f"{OUT_DIR}/configuration.json", "w"), indent=2)
print("All outputs saved in", OUT_DIR)